In [1]:
import cv2
import os
import numpy as np
from typing import List, Dict, Optional
from src.feature_extractor import FeatureExtractor
from src.detector import Detector
from src.tracker import Tracker
from src.gtalink import GTALink
from src.compute_metrics import compute_metrics
from src.utils import *

# Path & Config

In [ ]:
#BASE = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT"
BASE = r"D:\UIT 3rd year\NhanDang\Project"

# models
YOLO_PATH = os.path.join(BASE, r"PipelineMOT\models\best_yolo26.pt")
RF_PATH = os.path.join(BASE, r"PipelineMOT\models\best_rf_detr.pth")
SOLIDER_PATH = os.path.join(BASE, r"PipelineMOT\models\swin_small_converted.pth")

# data
VIDEOS_DIR = r"D:\UIT 3rd year\NhanDang\Project\Data\wide_view\videos"
GROUNDTRUTH_DIR = r"D:\UIT 3rd year\NhanDang\Project\Data\wide_view\annotations"
ALL_VIDEOS = sorted(os.listdir(VIDEOS_DIR))
TEST = [ALL_VIDEOS[18]]     # đang chạy cho video valid đầu tiên để tunning

# output
OUTPUT_DIR = os.path.join(BASE, r"Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# all config
ALL_DETECTOR  = ["yolo26", "rf_detr"]
ALL_EXTRACTOR = ["osnet", "solider", "color_histogram"]
ALL_TRACKER   = ["bytetrack", "ocsort", "strongsort", "deepeiou"]

In [3]:
def Testing(videos, detector, extractor, 
            tracker, refiner, extractor_refiner, 
            tracker_kwargs: Optional[Dict] = None, 
            visualize = False, output_video = None,
            input_tracks_map = None,  # dict {vid_name: all_tracks} để tái sử dụng
            **viz_kwargs):
    
    all_results = {}
    metrics_accum = {}
    # Thêm: dict lưu all_tracks mỗi video để tái sử dụng
    all_tracks_map = {}

    # Xác định tên các thành phần cho tên file
    det_name = detector.backend if detector is not None else "None"
    ext_name = extractor.backend if extractor is not None else "None"
    trk_name = tracker if tracker is not None else "None"
    ref_ext_name = (extractor_refiner.backend if extractor_refiner is not None else \
                    extractor.backend if extractor is not None else "None") \
                if refiner is not None else "None"

    result_filename = f"result_{det_name}_{ext_name}_{trk_name}_{ref_ext_name}.txt"
    result_path = os.path.join(OUTPUT_DIR, result_filename)

    for vid in videos:
        video_path = os.path.join(VIDEOS_DIR, vid)
        gt_path = os.path.join(GROUNDTRUTH_DIR, vid.split('.')[0] + ".csv")

        print(f"\n{'='*60}")
        print(f"Processing video: {vid}")
        print(f"{'='*60}")
        
        # Lấy cached tracks nếu có (tránh chạy lại MOT)
        input_tracks = input_tracks_map.get(vid) if input_tracks_map is not None else None

        result = run_pipeline(
            video_path      = video_path,
            gt_csv_path     = gt_path,
            detector        = detector,
            tracker_name    = tracker,
            output_path     = output_video,
            extractor       = extractor,
            refiner         = refiner,
            refiner_extractor = extractor_refiner,
            tracker_kwargs  = tracker_kwargs or {},
            visualize       = visualize,
            input_tracks = input_tracks,
            **viz_kwargs,
        )

        metrics = result['metrics']
        all_results[vid] = metrics
        # Lưu all_tracks theo tên video
        all_tracks_map[vid] = result['all_tracks']

        # Tích luỹ metrics để tính trung bình
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                metrics_accum.setdefault(k, []).append(v)

    # Tính trung bình
    avg_metrics = {k: np.mean(vals) for k, vals in metrics_accum.items()}

    print(f"\n{'='*60}")
    print("AVERAGE METRICS ACROSS ALL VIDEOS:")
    for k, v in avg_metrics.items():
        print(f"  {k}: {v:.4f}")
    print(f"{'='*60}")

    # Lưu kết quả ra file
    with open(result_path, 'w', encoding='utf-8') as f:
        f.write(f"Pipeline: {det_name} -> {ext_name} -> {trk_name} -> GTALink({ref_ext_name})\n")
        f.write(f"{'='*60}\n\n")

        for vid, metrics in all_results.items():
            f.write(f"[{vid}]\n")
            for k, v in metrics.items():
                f.write(f"  {k}: {v:.4f}\n" if isinstance(v, float) else f"  {k}: {v}\n")
            f.write("\n")

        f.write(f"{'='*60}\n")
        f.write("AVERAGE:\n")
        for k, v in avg_metrics.items():
            f.write(f"  {k}: {v:.4f}\n")

    print(f"\nKết quả đã lưu tại: {result_path}")

    return {
        'per_video':      all_results,
        'average':        avg_metrics,
        'all_tracks_map': all_tracks_map,  # {vid_name: all_tracks}
    }

In [4]:
detector_yolo26          = Detector(backend="yolo26", model_path=YOLO_PATH)
detector_rfdetr          = Detector(backend="rfdetr", model_path=RF_PATH)
extractor_color   = FeatureExtractor(backend="color_histogram")
extractor_osnet   = FeatureExtractor(backend="osnet")
extractor_solider = FeatureExtractor(
    backend="solider",
    solider_model_path=SOLIDER_PATH,
    solider_arch="swin_small",
    solider_semantic_weight=0.2,
)
refiner = GTALink(merge_dist_thres = 0.4)

[Detector] YOLO loaded on cpu


C:\Users\ACER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Successfully loaded imagenet pretrained weights from "C:\Users\ACER/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
[FeatureExtractor] OSNet loaded on cpu
Missing: 0 | Unexpected: 8
[FeatureExtractor] SOLIDER (swin_small) loaded on cpu
[FeatureExtractor] Feature dim: 768


# YOLO26

## OSNET

In [5]:
result_2 = Testing(TEST, detector_yolo26, extractor = None,
                   tracker = "bytetrack", refiner=refiner, extractor_refiner=extractor_solider,
                   input_tracks_map=None,
                   tracker_kwargs=None)  


Processing video: F_20220220_1_1800_1830.mp4

PIPELINE: YOLO26 → NONE → BYTETRACK → GTALink(solider)
===== Chưa có kết quả MOT - Chạy toàn bộ pipeline ======
Processing frame 0


KeyboardInterrupt: 

In [ ]:
# # yolo26 - osnet - deepeiou - None
# result_1 = Testing(TEST, detector_yolo26, extractor_osnet,
#                    "deepeiou", refiner=None, extractor_refiner=None,
#                    input_tracks_map = None, tracker_kwargs = None)

# # yolo26 - osnet - deepeiou - solider
# result_2 = Testing(TEST, detector_yolo26, extractor_osnet,
#                    "deepeiou", refiner=refiner, extractor_refiner=extractor_osnet,
#                    input_tracks_map=result_1['all_tracks_map'],
#                    tracker_kwargs=None)  

# RF-DETR